# Vector Search Load Test

Profile query latency against a selectable vector search index using two methods:

1. **Async Python** – concurrent `aiohttp` calls to the Vector Search REST API
2. **Spark SQL** – `vector_search()` table-valued function

Use the `index_name` widget to choose which index to benchmark. The notebook collects p50 / p90 / p99 summaries.

In [0]:
%pip install --quiet aiohttp databricks-sdk
%restart_python

In [0]:
import time
import asyncio
import aiohttp
import pandas as pd
import numpy as np
from databricks.sdk import WorkspaceClient

# --- Configuration ---
INDEX_NAME = dbutils.widgets.get("index_name")
SOURCE_TABLE = INDEX_NAME.removesuffix("_index")
CONCURRENCY = 50            # max parallel async requests
TOP_K = 10                  # results per query
WARMUP = 5                  # warmup queries (excluded from stats)

# --- Auth ---
w = WorkspaceClient()
HOST = w.config.host.rstrip("/")
ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
TOKEN = ctx.apiToken().get()

# --- Load query vectors from source table ---
query_vectors = (
    spark.table(SOURCE_TABLE)
    .select("query_vec")
    .toPandas()["query_vec"]
    .tolist()
)
print(f"Index: {INDEX_NAME}")
print(f"Source table: {SOURCE_TABLE}")
print(f"Loaded {len(query_vectors)} query vectors (dim={len(query_vectors[0])})")

## Method A: Async Python (aiohttp)

Fire oncurrent requests (bounded by `CONCURRENCY` semaphore) against the Vector Search REST API and record per-query wall-clock latency.

In [0]:
async def _query_one(session: aiohttp.ClientSession, sem: asyncio.Semaphore, vec, url, headers):
    """Issue a single vector search query and return latency in ms."""
    payload = {"query_vector": vec.tolist() if hasattr(vec, "tolist") else vec, "columns": ["registeredAccountId"], "num_results": TOP_K}
    async with sem:
        t0 = time.perf_counter()
        async with session.post(url, json=payload, headers=headers) as resp:
            await resp.json()
        return (time.perf_counter() - t0) * 1000


async def run_async_benchmark(vectors):
    url = f"{HOST}/api/2.0/vector-search/indexes/{INDEX_NAME}/query"
    headers = {"Authorization": f"Bearer {TOKEN}", "Content-Type": "application/json"}
    sem = asyncio.Semaphore(CONCURRENCY)

    async with aiohttp.ClientSession() as session:
        # Warmup
        warmup_tasks = [_query_one(session, sem, v, url, headers) for v in vectors[:WARMUP]]
        await asyncio.gather(*warmup_tasks)

        # Benchmark
        tasks = [_query_one(session, sem, v, url, headers) for v in vectors]
        latencies = await asyncio.gather(*tasks)

    return list(latencies)


# Run the async benchmark
async_batch_start = time.perf_counter()
async_latencies = await run_async_benchmark(query_vectors)
async_batch_ms = (time.perf_counter() - async_batch_start) * 1000
print(f"Completed {len(async_latencies)} queries")
print(f"  total_batch_ms={async_batch_ms:.1f}ms")
print(f"  avg_per_query_ms={async_batch_ms / max(len(async_latencies), 1):.1f}ms")
print(f"  p50={np.percentile(async_latencies, 50):.1f}ms  "
      f"p90={np.percentile(async_latencies, 90):.1f}ms  "
      f"p99={np.percentile(async_latencies, 99):.1f}ms")

## Method B: Spark SQL `vector_search()`

Run vector search queries in one SQL call via the `vector_search()` table-valued function and record driver-side wall-clock latency for the batch.

In [0]:
RESULTS_TABLE = f"{SOURCE_TABLE}_results"
print(f"Running Spark SQL benchmark: LATERAL vector_search over {SOURCE_TABLE}")
print(f"Materializing results to {RESULTS_TABLE}")

t0 = time.perf_counter()
spark.sql(f"""
    CREATE OR REPLACE TABLE {RESULTS_TABLE} AS
    SELECT src.registeredAccountId AS query_id, vs.registeredAccountId AS neighbor_id, vs.search_score
    FROM {SOURCE_TABLE} src,
    LATERAL vector_search(
        index => '{INDEX_NAME}',
        query_vector => src.query_vec,
        num_results => {TOP_K}
    ) AS vs
""")
spark_batch_ms = (time.perf_counter() - t0) * 1000

stats = spark.sql(f"""
    SELECT
        count(*) AS total_rows,
        count(DISTINCT query_id) AS num_queries,
        avg(search_score) AS avg_score,
        percentile_approx(search_score, 0.5) AS p50_score,
        percentile_approx(search_score, 0.1) AS p10_score,
        min(search_score) AS min_score
    FROM {RESULTS_TABLE}
""").first()

spark_query_count = stats["num_queries"]
spark_latencies = [spark_batch_ms]

print(f"Completed {spark_query_count} queries, {stats['total_rows']} result rows")
print(f"  total_batch_ms={spark_batch_ms:.1f}ms")
print(f"  avg_per_query_ms={spark_batch_ms / max(spark_query_count, 1):.1f}ms")
print(f"  avg_score={stats['avg_score']:.4f}  p50_score={stats['p50_score']:.4f}  min_score={stats['min_score']:.4f}")
display(spark.table(RESULTS_TABLE).limit(20))

## Results & Comparison

In [0]:
# Build results DataFrame
results = pd.DataFrame({
    "method": ["async_python"] * len(async_latencies) + ["spark_sql"] * len(spark_latencies),
    "query_idx": list(range(len(async_latencies))) + list(range(len(spark_latencies))),
    "latency_ms": async_latencies + spark_latencies,
})

batch_runtime_ms = pd.DataFrame({
    "method": ["async_python", "spark_sql"],
    "batch_runtime_ms": [async_batch_ms, spark_batch_ms],
})

# Summary statistics
summary = (
    results.groupby("method")["latency_ms"]
    .agg(
        count="count",
        mean="mean",
        p50=lambda s: s.quantile(0.50),
        p90=lambda s: s.quantile(0.90),
        p99=lambda s: s.quantile(0.99),
        max="max",
    )
    .round(2)
    .reset_index()
    .merge(batch_runtime_ms, on="method", how="left")
)

summary["avg_batch_ms_per_query"] = (summary["batch_runtime_ms"] / summary["count"]).round(2)
summary["qps"] = (summary["count"] / (summary["batch_runtime_ms"] / 1000)).round(1)

print(summary.to_string(index=False))
display(spark.createDataFrame(summary))